First attempt at making raster from LAZ using binned statistic 

In [ ]:
import numpy as np
from scipy.stats import binned_statistic_2d
import rasterio
from rasterio.transform import from_origin
# 1) keep only ground points
ground_mask = clipped_las.classification == 2

x = clipped_las.x[ground_mask]
y = clipped_las.y[ground_mask]
z = clipped_las.z[ground_mask]

print("Number of ground points:", len(z))

# 2) choose raster cell size
cell_size = 1.0  # meters

# 3) define raster extent
xmin, xmax = x.min(), x.max()
ymin, ymax = y.min(), y.max()

# make edges that align to cell size
x_edges = np.arange(xmin, xmax + cell_size, cell_size)
y_edges = np.arange(ymin, ymax + cell_size, cell_size)

# 4) bin points into cells
# for a bare-earth DEM, min is often a reasonable starting choice
dem, x_edge_out, y_edge_out, binnumber = binned_statistic_2d(
    x, y, z,
    statistic="min",
    bins=[x_edges, y_edges]
)
# scipy returns array indexed as [xbin, ybin], so transpose for raster layout
dem = dem.T

# flip vertically so row 0 is the top of the raster
dem = np.flipud(dem)

# 5) build affine transform
transform = from_origin(x_edges[0], y_edges[-1], cell_size, cell_size)

# 6) write GeoTIFF
with rasterio.open(
    "parcel_dem.tif",
    "w",
    driver="GTiff",
    height=dem.shape[0],
    width=dem.shape[1],
    count=1,
    dtype=dem.dtype,
    crs=clipped_las.header.parse_crs(),
    transform=transform,
    nodata=np.nan
) as dst:
    dst.write(dem, 1)
    with rasterio.open("parcel_dem.tif") as src:
    dem = src.read(1)

# mask NaNs so they show as white
dem_masked = np.ma.masked_invalid(dem)

fig, ax = plt.subplots()

cmap = plt.cm.terrain
cmap.set_bad(color='white')  # missing data = white

im = ax.imshow(dem_masked, cmap=cmap)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Elevation (meters)")

ax.set_title("DEM (Elevation)")
plt.show()

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

with rasterio.open("parcel_dem.tif") as src:
    dem = src.read(1)

# mask NaNs so they show as white
dem_masked = np.ma.masked_invalid(dem)

fig, ax = plt.subplots()

cmap = plt.cm.terrain
cmap.set_bad(color='white')  # missing data = white

im = ax.imshow(dem_masked, cmap=cmap)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Elevation (meters)")

ax.set_title("DEM (Elevation)")
plt.show()